In [1]:
import os
os.environ["PYTHONNET_RUNTIME"] = "coreclr"
os.environ["DOTNET_SYSTEM_DRAWING_USE_GDIPLUS"] = "1"

import clr
import numpy as np
import time
from datetime import timedelta
from pythonnet import load
from System import Environment, Array, String

# Load .NET Core runtime
load("coreclr")

# Try to import System.IO; fallback to os.chdir if it fails
try:
    from System.IO import Directory, Path, File
    use_dotnet_dir = True
except Exception as e:
    print(f"Note: System.IO import failed ({e}), using os.chdir instead")
    import os
    use_dotnet_dir = False

# Define DWSIM path
dwsimpath = "/usr/local/lib/dwsim/"

# Set working directory
if use_dotnet_dir:
    Directory.SetCurrentDirectory(dwsimpath)
else:
    os.chdir(dwsimpath)

# Add DWSIM assemblies
clr.AddReference(dwsimpath + "CapeOpen.dll")
clr.AddReference(dwsimpath + "DWSIM.Automation.dll")
clr.AddReference(dwsimpath + "DWSIM.Interfaces.dll")
clr.AddReference(dwsimpath + "DWSIM.GlobalSettings.dll")
clr.AddReference(dwsimpath + "DWSIM.SharedClasses.dll")
clr.AddReference(dwsimpath + "DWSIM.Thermodynamics.dll")
clr.AddReference(dwsimpath + "DWSIM.UnitOperations.dll")
clr.AddReference(dwsimpath + "DWSIM.Inspector.dll")
clr.AddReference(dwsimpath + "System.Buffers.dll")
clr.AddReference(dwsimpath + "DWSIM.Thermodynamics.ThermoC.dll")

# Now import DWSIM types
from DWSIM.Interfaces.Enums.GraphicObjects import ObjectType
from DWSIM.Thermodynamics import Streams, PropertyPackages
from DWSIM.UnitOperations import UnitOperations
from DWSIM.Automation import Automation3
from DWSIM.GlobalSettings import Settings

print("DWSIM imports successful!")

DWSIM imports successful!


In [2]:
# Create an instance of the Automation3 class from the DWSIM.Automation module
# This class provides methods for automating tasks in DWSIM, such as creating and manipulating flowsheets
interf = Automation3()

In [3]:
# Creates a flowsheet
sim = interf.CreateFlowsheet()

In [4]:
# Add Compounds
sim.AddCompound("Water")
sim.AddCompound("Methanol")

In [5]:
one = sim.AddFlowsheetObject('Material Stream','1')
two = sim.AddFlowsheetObject('Material Stream','2')
three = sim.AddFlowsheetObject('Material Stream','3')
four = sim.AddFlowsheetObject('Material Stream','4')
HEX_1 = sim.AddFlowsheetObject('Heat Exchanger','HEX-1')

In [6]:
one = one.GetAsObject()
two = two.GetAsObject()
three = three.GetAsObject()
four = four.GetAsObject()
HEX_1 = HEX_1.GetAsObject()

In [7]:
sim.ConnectObjects(one.GraphicObject , HEX_1.GraphicObject, -1, -1)
sim.ConnectObjects(HEX_1.GraphicObject , two.GraphicObject, -1, -1)
sim.ConnectObjects(three.GraphicObject , HEX_1.GraphicObject, -1, -1)
sim.ConnectObjects(HEX_1.GraphicObject , four.GraphicObject, -1, -1)

In [8]:
sim.AutoLayout()

In [9]:
one.SetOverallComposition(Array[float]([0.99, 0.01]))
one.SetTemperature(300.0) # K
one.SetPressure(101325.0) # Pa
one.SetMassFlow(1.0) # kg/s

three.SetOverallComposition(Array[float]([0.0, 1.0]))
three.SetTemperature(300.0) # K
three.SetPressure(101325.0) # Pa
three.SetMassFlow(1.0) # kg/s

'3: mass flow set to 1 kg/s'

In [10]:
# property package
Thermo_Package = sim.CreateAndAddPropertyPackage("NRTL")

In [11]:
# Calc Modes
HEX_1.CalculationMode = HEX_1.CalculationMode.CalcBothTemp_UA

# Setting properties
HEX_1.Area = 1.0 # m2
HEX_1.OverallCoefficient = 1000 # W/m2-K
HEX_1.HotSidePressureDrop = 0 # Pa
HEX_1.ColdSidePressureDrop = 0 # Pa
HEX_1.FlowDir.CoCurrent


<FlowDirection.CoCurrent: 1>

In [12]:
# request a calculation
errors = interf.CalculateFlowsheet4(sim)
list(errors)

[Exception('HEX-1: failed to calculate the Number of Transfer Units (NTU) with the current input and specs')]

In [13]:
# save file

fileNameToSave = Path.Combine(Environment.GetFolderPath(Environment.SpecialFolder.Desktop), "/workspace/00 FlowSheet Automation/06 HEX Automation/06 HEX.dwxmz")

interf.SaveFlowsheet(sim, fileNameToSave, True)